# TiA 2 — Generalisation vs Memorisation

Test how capacity, regularisation and distribution shift affect fitting and generalisation.

## How to work through this activity

This is a guided investigation rather than a coding tutorial. For each experiment:

1. Read the mathematical claim and identify the quantity being measured.
2. Predict the qualitative result before running the code.
3. Run one cell at a time and inspect both values and plots.
4. Change only the suggested variable; rerun and explain what changed.
5. Answer the **Explain** questions in your own words.

The code contains more comments than production software intentionally. You are not expected to memorise framework syntax. Focus on the relationship between assumptions, measurements and conclusions.

## Notation and prediction

Given training sample $S=\{(x_i,y_i)\}_{i=1}^n$, empirical risk is

$$\hat R_S(f)=\frac1n\sum_{i=1}^n\ell(f(x_i),y_i),$$

whereas population risk is $R(f)=\mathbb E_{(x,y)\sim p_{\mathrm{data}}}[\ell(f(x),y)]$. Their difference is the generalisation gap. Parameter count measures capacity only imperfectly, and interpolation ($\hat R_S\approx0$) does not imply $R(f)\approx0$. Predict what will happen when $y_i$ is replaced by a fixed random permutation.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Fix every random-number generator so that your plots match the reference run.
# After completing the guided activity, change the seed to test robustness.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# The default path is designed for a CPU. Set this to False only after the
# notebook works and you want to run longer variants.
FAST_MODE = True

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
X,y=load_digits(return_X_y=True); X=X/16
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.35,stratify=y,random_state=7)
# A smaller training sample makes exact memorisation observable on a laptop.
Xtr,ytr=Xtr[:600],ytr[:600]

def fit(width,labels,alpha=0.0):
    # L-BFGS is used to study attainable fits, not stochastic training dynamics.
    m=MLPClassifier((width,),alpha=alpha,solver="lbfgs",max_iter=300 if FAST_MODE else 800,random_state=7).fit(Xtr,labels)
    return m,accuracy_score(labels,m.predict(Xtr)),accuracy_score(yte,m.predict(Xte))

rows=[]
random_y=rng.permutation(ytr)
for width in [8,32,64]:
    for name,labels in [("true",ytr),("random",random_y)]:
        m,tr,te=fit(width,labels); rows.append((width,name,sum(w.size for w in m.coefs_),tr,te))
print("width labels parameters train_acc test_acc")
for r in rows: print(f"{r[0]:5} {r[1]:6} {r[2]:10} {r[3]:.3f} {r[4]:.3f}")
widest_random=[r for r in rows if r[0]==64 and r[1]=="random"][0]
widest_true=[r for r in rows if r[0]==64 and r[1]=="true"][0]
assert widest_random[3] > .95 and widest_random[4] < .20
assert widest_true[4] > .85

## Regularisation and shift

Weight decay changes the objective; early stopping changes the optimisation path; augmentation changes the empirical distribution. They need not have equivalent effects.

In [ ]:
model,_,_=fit(128,ytr,alpha=1e-2)
# Rotate every 8×8 image using NumPy. This changes the test distribution
# without introducing another software dependency.
shifted=np.rot90(Xte.reshape(-1,8,8),k=1,axes=(1,2)).reshape(-1,64)
print({"in_distribution":accuracy_score(yte,model.predict(Xte)),"rotated":accuracy_score(yte,model.predict(shifted))})
for alpha in [0,1e-3,1e-1,1]:
    _,tr,te=fit(128,ytr,alpha); print(f"weight_decay={alpha:g}: train={tr:.3f}, test={te:.3f}")
assert accuracy_score(yte,model.predict(shifted)) < accuracy_score(yte,model.predict(Xte))-.05

### Explain

1. Does interpolation imply generalisation? Use the random-label control.
2. Why is the rotated set not merely a noisier estimate of the same test accuracy?
3. Identify one conclusion this small experiment cannot justify.

**Reading:** [Zhang et al., Understanding Deep Learning Requires Rethinking Generalization](https://openreview.net/forum?id=Sy8gdB9xx).

## Expected pattern and limits

The wider model should fit arbitrary labels while remaining near chance on genuine test labels. Genuine labels should generalise, and rotation should reduce accuracy. Regularisation need not improve this already-small clean problem. The experiment demonstrates possibility—not a complete theory of neural-network generalisation.